In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from datetime import datetime

# Load the dataset
file_path = 'cluster0_df.csv'
df = pd.read_csv(file_path)

# Parse dates and sort the dataset
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df.sort_values('Order_Date', inplace=True)

# Extract the target variable
data = df['Number of Orders'].values

# Normalize the data
scaler = MinMaxScaler(feature_range=(0, 1))
data_normalized = scaler.fit_transform(data.reshape(-1, 1))

# Prepare sequences
def create_sequences(data, seq_length):
    sequences = []
    labels = []
    for i in range(len(data) - seq_length):
        seq = data[i:i + seq_length]
        label = data[i + seq_length]
        sequences.append(seq)
        labels.append(label)
    return np.array(sequences), np.array(labels)

seq_length = 5  # Use the last 5 days to predict the next day's orders
X, y = create_sequences(data_normalized, seq_length)

# Convert to PyTorch tensors
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

class TimeSeriesDataset(Dataset):
    def __init__(self, features, targets):
        self.features = features
        self.targets = targets

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]

dataset = TimeSeriesDataset(X, y)
train_loader = DataLoader(dataset, batch_size=4, shuffle=True)

# Define the LSTM model
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h, _ = self.lstm(x)
        h = h[:, -1, :]  # Take the output of the last time step
        out = self.fc(h)
        return out

# Model parameters
input_size = 1
hidden_size = 32
num_layers = 2
output_size = 1

model = LSTMModel(input_size, hidden_size, num_layers, output_size)

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 50
for epoch in range(num_epochs):
    for inputs, targets in train_loader:
        inputs = inputs.unsqueeze(-1)  # Add input_size dimension (to match input_size=1)
        outputs = model(inputs)  # Correctly pass 3D tensor to LSTM
        loss = criterion(outputs, targets)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}")

# Making predictions for the entire dataset
model.eval()
predictions = []
with torch.no_grad():
    for i in range(len(X)):
        seq = X[i].unsqueeze(0).unsqueeze(-1)
        pred = model(seq)
        predictions.append(pred.item())

# Rescale predictions back to the original scale
predictions_rescaled = scaler.inverse_transform(np.array(predictions).reshape(-1, 1))
actual_rescaled = scaler.inverse_transform(y.numpy().reshape(-1, 1))

# Plotting the results
plt.figure(figsize=(10, 6))
plt.plot(df['Order_Date'][seq_length:], actual_rescaled, label="Actual Orders", color="blue")
plt.plot(df['Order_Date'][seq_length:], predictions_rescaled, label="Predicted Orders", color="orange")
plt.xlabel("Date")
plt.ylabel("Number of Orders")
plt.title("Actual vs Predicted Orders")
plt.legend()
plt.grid(True)
plt.show()

# Predict the next day's orders
with torch.no_grad():
    last_sequence = X[-1].unsqueeze(0).unsqueeze(-1)
    next_day_prediction = model(last_sequence)
    next_day_prediction = scaler.inverse_transform(next_day_prediction.numpy())
    print(f"Predicted orders for the next day: {next_day_prediction[0][0]}")


ValueError: LSTM: Expected input to be 2D or 3D, got 4D instead